# Clean and Prepare Data for Analysis with Python

This interactive course develops a complete preprocessing workflow for scientific signals. You will diagnose raw observations, detect local outliers, handle missing values, interpolate small gaps, normalize measurements, remove a slow trend, smooth noise, and validate the final dataset.

**Estimated time:** 60–90 minutes  
**Prerequisites:** Python variables, arrays, functions, and basic plotting  
**Environment:** use `/Users/victorbui/venvs/ai312`

Run the cells in order. Each **Practice** cell is followed by a checkable solution. The course uses a reproducible light curve created locally, so it requires no download.

## Learning outcomes

After this course, you can:

- distinguish missing values, outliers, noise, and long-term trends;
- choose among ignoring, removing, and replacing missing observations;
- interpolate by real sample locations and limit filling to defensible gaps;
- compare centering, z-scores, relative scaling, and range scaling;
- fit a polynomial baseline without letting the event of interest control the fit;
- compare centered, trailing, and leading smoothing windows;
- record validation checks before exporting prepared data.

In [ ]:
# Load the libraries and reusable functions supplied with the course.
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from Python_Clean_Prepare_Data_Course import (
    add_normalizations,
    create_light_curve,
    interpolate_small_gaps,
    polynomial_detrend,
    rolling_mad_outliers,
    standardize_missing,
    time_window_smooth,
)

COURSE_DIR = Path.cwd()
OUTPUT_DIR = COURSE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 12)

## 1. Diagnose the raw data

Cleaning starts with evidence. Check the table shape, column meanings, order, types, missing values, impossible codes, and plots before choosing a method. A measurement equal to `-999` is a sentinel code; it must become a missing value before summary statistics are trustworthy.

In [ ]:
# Create the reproducible observations and inspect them before modification.
raw = create_light_curve(seed=42)
display(raw.head())
print(raw.info())
print(raw["brightness"].describe())
print("Explicit missing values:", raw["brightness"].isna().sum())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(raw["time_days"], raw["brightness"], lw=1)
ax.set(xlabel="Time (days)", ylabel="Relative brightness", title="Raw light curve")
plt.show()

### Practice 1

Find every row whose brightness is below zero. Explain why the ordinary minimum and mean cannot describe this signal correctly yet.

In [ ]:
# Practice 1: write a Boolean filter for impossible brightness values.
# impossible = ...
# display(impossible)

In [ ]:
# Solution 1: identify the sentinel code before computing statistics.
impossible = raw.loc[raw["brightness"] < 0]
display(impossible)
assert impossible["brightness"].eq(-999).all()

## 2. Locate and standardize missing values

Numeric missing values use `NaN`. `Series.isna()` locates them, `sum()` counts them, and most pandas reductions skip them by default. Decide whether a sentinel such as `-999` means missing from documentation or domain knowledge; never infer that meaning from magnitude alone.

Three common strategies have different consequences:

1. **Ignore during a calculation:** preserves observations but requires functions with known missing-value behavior.
2. **Remove rows:** simple, but it changes the sample count and can hide time gaps in line plots.
3. **Replace values:** preserves alignment, but filled values are estimates rather than measurements.

In [ ]:
# Convert the documented sentinel to NaN and compare missing-value strategies.
data = standardize_missing(raw, "brightness", sentinels=[-999])
missing_mask = data["brightness"].isna()
print("Missing values:", int(missing_mask.sum()))
print("Mean that skips NaN:", data["brightness"].mean())
print("Rows after removal:", len(data.dropna(subset=["brightness"])))

median_filled = data["brightness"].fillna(data["brightness"].median())
print("Missing after median replacement:", int(median_filled.isna().sum()))

### Practice 2

Create `missing_rows` with the time and brightness for every missing observation. Then calculate the fraction of rows that are missing.

In [ ]:
# Practice 2: select the missing rows and calculate their fraction.
# missing_rows = ...
# missing_fraction = ...

In [ ]:
# Solution 2: retain the row context when reporting missingness.
missing_rows = data.loc[data["brightness"].isna(), ["time_days", "brightness"]]
missing_fraction = data["brightness"].isna().mean()
print("Missing fraction:", f"{missing_fraction:.2%}")
assert len(missing_rows) == data["brightness"].isna().sum()

## 3. Detect local outliers

An outlier is unusual relative to an expected distribution or local pattern. A global rule can confuse slow drift with unusual points. Here, a rolling median estimates the local center and a rolling median absolute deviation estimates local variation. The threshold is a modeling choice, so review detected points visually.

For a measured signal, marking outliers as missing often preserves the time grid better than deleting rows.

In [ ]:
# Detect isolated spikes with a robust local rule.
outlier_mask = rolling_mad_outliers(data["brightness"], window=41, threshold=5)
print("Detected outliers:", int(outlier_mask.sum()))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(data["time_days"], data["brightness"], lw=1, label="Input")
ax.scatter(
    data.loc[outlier_mask, "time_days"],
    data.loc[outlier_mask, "brightness"],
    color="crimson", marker="x", s=50, label="Detected outlier"
)
ax.set(xlabel="Time (days)", ylabel="Relative brightness")
ax.legend()
plt.show()

cleaned = data.copy()
cleaned.loc[outlier_mask, "brightness"] = np.nan

### Practice 3

Run the detector with thresholds 3, 5, and 8. Record the detected count for each threshold. Which setting best isolates the visible spikes without flagging the transit dips?

In [ ]:
# Practice 3: compare how sensitivity changes with the threshold.
# threshold_counts = {...}
# threshold_counts

In [ ]:
# Solution 3: larger thresholds require a more extreme local deviation.
threshold_counts = {
    threshold: int(rolling_mad_outliers(data["brightness"], 41, threshold).sum())
    for threshold in (3, 5, 8)
}
threshold_counts

## 4. Interpolate missing data

Nearest-neighbor interpolation produces steps. Linear interpolation connects nearby known values with straight segments. When observations are unevenly spaced, interpolation must use the real sample locations.

Gap size matters. A short gap may have enough neighboring information for a defensible estimate. A long gap can become an unsupported straight bridge. The course fills no more than five consecutive missing rows and leaves the long gap visible.

In [ ]:
# Compare index-position interpolation with interpolation over real x locations.
x = pd.Series([0, 2, 3, 4, 7, 9, 10, 12, 15], dtype=float)
y = pd.Series([0, np.nan, 7, 8, np.nan, 2, -3, np.nan, -8], dtype=float)
by_position = y.interpolate(method="linear")
by_location = pd.Series(y.to_numpy(), index=x).interpolate(method="index")

comparison = pd.DataFrame({
    "x": x,
    "observed": y,
    "equal_spacing": by_position,
    "actual_spacing": by_location.to_numpy(),
})
display(comparison)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, y, "s", ms=7, label="Observed")
ax.plot(x, by_position, "o--", label="Equal spacing assumed")
ax.plot(x, by_location.to_numpy(), "*-", label="Actual x locations")
ax.set(xlabel="Sample location", ylabel="Value")
ax.legend()
plt.show()

In [ ]:
# Fill short internal gaps in the light curve and retain the long gap.
interpolated = interpolate_small_gaps(cleaned, "brightness", limit=5)
print("Missing before:", int(cleaned["brightness"].isna().sum()))
print("Missing after:", int(interpolated["brightness"].isna().sum()))
print("Long gap retained:", interpolated.iloc[355:377]["brightness"].isna().all())

### Practice 4

Repeat the interpolation with `limit=25`. Compare rows 355–376. Why could this result mislead a later transit analysis?

In [ ]:
# Practice 4: deliberately fill the large gap and inspect the result.
# overfilled = ...
# display(overfilled.iloc[350:382])

In [ ]:
# Solution 4: a permissive limit creates estimates across the long gap.
overfilled = interpolate_small_gaps(cleaned, "brightness", limit=25)
print("Missing in long gap after limit=25:", int(overfilled.iloc[355:377]["brightness"].isna().sum()))
# A line across a long interval looks continuous even though no measurements exist.

## 5. Shift and scale data

Normalization should match the analytical goal:

- **Centering** subtracts the mean but retains the original unit and spread.
- **Z-scores** express distance from the mean in standard deviations.
- **Relative scaling** divides centered measurements by their mean, producing fractional change.
- **Range scaling** maps the observed minimum and maximum to a chosen interval.

Fit scaling parameters on training data only when preparing machine-learning inputs; applying statistics from the complete dataset can leak information from evaluation data.

In [ ]:
# Add four transformations while preserving missing rows.
interpolated = interpolated.rename(columns={"brightness": "brightness_clean"})
normalized = add_normalizations(interpolated, "brightness_clean")
display(normalized.head())
print("z-score mean:", normalized["z_score"].mean())
print("z-score population standard deviation:", normalized["z_score"].std(ddof=0))
print("range-scaled minimum and maximum:", normalized["scaled_minus1_to1"].min(), normalized["scaled_minus1_to1"].max())

### Practice 5

Create a two-column sensor table whose columns have very different units. Compute z-scores column by column and verify that both transformed columns have mean 0 and population standard deviation 1.

In [ ]:
# Practice 5: normalize each sensor independently.
sensors = pd.DataFrame({
    "temperature_c": [18.0, 19.5, 21.0, 22.5, 24.0],
    "pressure_pa": [100800, 101000, 101300, 101600, 101900],
})
# sensor_z = ...

In [ ]:
# Solution 5: pandas aligns the mean and standard deviation by column.
sensor_z = (sensors - sensors.mean()) / sensors.std(ddof=0)
display(sensor_z)
assert np.allclose(sensor_z.mean(), 0)
assert np.allclose(sensor_z.std(ddof=0), 1)

## 6. Remove an unwanted trend

Detrending fits a baseline and subtracts it from the observations. A polynomial degree controls flexibility: too low leaves structure; too high may absorb the signal of interest. Exclude known event regions from the baseline fit when they should remain in the residual.

Time is centered and scaled before polynomial fitting to improve numerical conditioning.

In [ ]:
# Exclude the two transit regions from the cubic baseline fit.
time = normalized["time_days"]
baseline_mask = ~(time.between(1.80, 2.20) | time.between(3.98, 4.32))
detrended = polynomial_detrend(
    normalized, "relative_brightness", degree=3, mask=baseline_mask
)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(time, normalized["relative_brightness"], lw=1, label="Normalized")
axes[0].plot(time, detrended["trend"], lw=2, label="Fitted trend")
axes[0].set(ylabel="Fractional change")
axes[0].legend()
axes[1].plot(time, detrended["detrended"], lw=1)
axes[1].axhline(0, color="black", lw=0.7)
axes[1].set(xlabel="Time (days)", ylabel="Residual")
plt.show()

### Practice 6

Fit degrees 1, 3, and 8. Calculate the root mean squared residual on baseline rows for each. Inspect the plots before deciding which degree is scientifically reasonable; the smallest residual does not automatically give the best model.

In [ ]:
# Practice 6: compare polynomial flexibility on baseline observations.
# degree_rmse = {}
# for degree in (1, 3, 8):
#     ...
# degree_rmse

In [ ]:
# Solution 6: measure fit error, then pair the number with visual review.
degree_rmse = {}
for degree in (1, 3, 8):
    candidate = polynomial_detrend(
        normalized, "relative_brightness", degree=degree, mask=baseline_mask
    )
    residual = candidate.loc[baseline_mask, "detrended"].dropna()
    degree_rmse[degree] = float(np.sqrt(np.mean(residual**2)))
degree_rmse

## 7. Smooth noisy data

A moving mean reduces random variation but blurs sharp features as its window grows. A moving median resists isolated spikes. Centered windows use past and future observations. A trailing window uses only current and earlier values, which suits real-time monitoring. A leading window uses current and later values and is useful for retrospective comparisons.

The helper below defines windows in days, so uneven sample spacing does not change their physical width.

In [ ]:
# Compare centered and trailing smoothing windows on the detrended signal.
detrended["centered_mean"] = time_window_smooth(
    detrended, "detrended", before_days=0.05, after_days=0.05, method="mean"
)
detrended["trailing_median"] = time_window_smooth(
    detrended, "detrended", before_days=0.10, after_days=0, method="median"
)
detrended["leading_median"] = time_window_smooth(
    detrended, "detrended", before_days=0, after_days=0.10, method="median"
)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(time, detrended["detrended"], color="0.75", lw=0.8, label="Detrended")
ax.plot(time, detrended["centered_mean"], lw=1.5, label="Centered mean")
ax.plot(time, detrended["trailing_median"], lw=1.2, label="Trailing median")
ax.set(xlabel="Time (days)", ylabel="Residual", xlim=(1.65, 2.35))
ax.legend()
plt.show()

In [ ]:
# Demonstrate how a larger window suppresses noise and signal detail.
generator = np.random.default_rng(123)
sample_x = np.arange(0, 8.1, 0.1)
sample_y = pd.Series(np.sin(sample_x) + generator.normal(0, 0.1, len(sample_x)))
mean_3 = sample_y.rolling(3, center=True, min_periods=1).mean()
mean_17 = sample_y.rolling(17, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sample_x, sample_y, "o-", ms=3, alpha=0.45, label="Noisy signal")
ax.plot(sample_x, mean_3, lw=2, label="3-point mean")
ax.plot(sample_x, mean_17, lw=2, label="17-point mean")
ax.set(xlabel="x", ylabel="y")
ax.legend()
plt.show()

### Practice 7

At one time point near the first transit, compare the centered, trailing, and leading medians. Explain which windows can use measurements that would be unavailable during live monitoring.

In [ ]:
# Practice 7: inspect the three window results near day 2.
# row_near_two = ...
# display(...)

In [ ]:
# Solution 7: a trailing window is causal because it does not look ahead.
row_near_two = (detrended["time_days"] - 2).abs().idxmin()
display(detrended.loc[[row_near_two], [
    "time_days", "detrended", "centered_mean", "trailing_median", "leading_median"
]])

## 8. Validate and export the prepared data

A clean-looking plot does not prove a valid transformation. Write explicit checks for row preservation, time order, sentinel removal, intended missingness, scaling, and output schema. Keep an audit table with counts and decisions so another analyst can review the workflow.

In [ ]:
# Select the deliverable columns and run targeted validation checks.
final_data = detrended.rename(columns={"centered_mean": "smoothed"})[[
    "time_days",
    "brightness_clean",
    "relative_brightness",
    "trend",
    "detrended",
    "smoothed",
    "trailing_median",
]].copy()

assert len(final_data) == len(raw)
assert final_data["time_days"].is_monotonic_increasing
assert not (final_data["brightness_clean"] == -999).any()
assert final_data.iloc[355:377]["brightness_clean"].isna().all()
assert abs(normalized["z_score"].mean()) < 1e-10
assert abs(normalized["z_score"].std(ddof=0) - 1) < 1e-10

quality_audit = pd.DataFrame({
    "check": [
        "input rows",
        "sentinel codes replaced",
        "outliers marked missing",
        "missing after short-gap interpolation",
        "rows retained in long gap",
    ],
    "value": [
        len(raw),
        int((raw["brightness"] == -999).sum()),
        int(outlier_mask.sum()),
        int(final_data["brightness_clean"].isna().sum()),
        int(final_data.iloc[355:377]["brightness_clean"].isna().sum()),
    ],
})
display(quality_audit)

In [ ]:
# Export the prepared observations and audit table beside the course.
final_data.to_csv(OUTPUT_DIR / "prepared_light_curve.csv", index=False)
quality_audit.to_csv(OUTPUT_DIR / "quality_audit.csv", index=False)
print("Saved:", OUTPUT_DIR / "prepared_light_curve.csv")
print("Saved:", OUTPUT_DIR / "quality_audit.csv")

## Final project

Create a reusable function that accepts a DataFrame with `time_days` and `brightness`. It should:

1. replace documented sentinels;
2. flag local outliers and mark them missing;
3. fill only short internal gaps;
4. calculate relative brightness;
5. fit a cubic trend while excluding supplied event intervals;
6. add centered and trailing smooths;
7. return the prepared table and a quality audit.

Test the function on a new light curve created with `seed=7`. Confirm that it keeps the input row count and leaves the long gap missing. The standalone course script provides a complete working pipeline to compare with your solution.

## Quick reference

| Goal | Python pattern |
|---|---|
| Locate missing values | `series.isna()` |
| Count missing values | `series.isna().sum()` |
| Standardize a sentinel | `series.replace(-999, np.nan)` |
| Remove missing rows | `df.dropna(subset=["value"])` |
| Fill with a statistic | `series.fillna(series.median())` |
| Linear interpolation by x | `pd.Series(y, index=x).interpolate(method="index")` |
| Center | `x - x.mean()` |
| Z-score | `(x - x.mean()) / x.std(ddof=0)` |
| Scale to [0, 1] | `(x - x.min()) / (x.max() - x.min())` |
| Polynomial fit | `np.polyfit(x, y, degree)` |
| Polynomial values | `np.polyval(coefficients, x)` |
| Centered moving mean | `series.rolling(5, center=True).mean()` |
| Trailing moving median | `series.rolling(4).median()` |
| Validate a condition | `assert condition` |
| Export a table | `df.to_csv("prepared.csv", index=False)` |

You have completed the course when you can justify every replacement or removal, preserve important time gaps, and explain how normalization, detrending, and smoothing change the interpretation of the signal.